In [49]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from pathlib import Path
import datetime as dt

CWD = Path(os.getcwd())
if 'notebooks' in str(CWD):
	sys.path.append(str(CWD.parent))
else:
	sys.path.append(str(CWD))


In [2]:
from src_strategy.utils.utils import load_df
from src_strategy.configs.pulmonarydysfunction import PulmonaryDysfunctionConfig
from src_strategy.configs.dataconfig import input_output_config_3_1

In [3]:
df_all = load_df(input_output_config_3_1.output_path, 'df_all.parquet')

In [4]:
config = PulmonaryDysfunctionConfig()

In [5]:
config.__dict__

{'encounter_col': 'EncounterEpicCsn',
 'event_dt_col': 'Event_DateTime',
 'event_name_col': 'Event_Name',
 'grouper_col': 'Event_Grouper',
 'type_col': 'Type',
 'val_col': 'NumericValue',
 'raw_val_col': 'Value',
 'BASELINE_CREATININE': 'Baseline_Creatinine',
 'BASELINE_SBP': 'Baseline_SBP',
 'BASELINE_RESP_RATE': 'Baseline_RespiratoryRate',
 'BASELINE_PULSE_RATE': 'Baseline_PulseRate',
 'BASELINE_PLATELETS': 'Baseline_Platelets',
 'BASELINE_BILIRUBIN': 'Baseline_Bilirubin',
 'BASELINE_eGFR': 'Baseline_eGFR',
 'BASELINE_WBC': 'Baseline_WBC',
 'vent_onoff': VentOnOffConfig(grouper='Vent On/Off', start_values=['On Going Hospital Vent', 'Initial', '$ On Going Hospital Vent'], termination_values=['Standby', '$ Extubation'], null_value_terminates=True, start_flag_col='vent_onoff_start_flag', termination_flag_col='vent_onoff_termination_flag'),
 'vent_documentation': VentDocumentationConfig(start_grouper='Vent on Documentation', termination_grouper='Vent off Documentation', start_flag_col='v

In [6]:
ventonoff_config = config.vent_onoff
ventdoc_config = config.vent_documentation
vento2_config = config.o2_delivery
exclusion_config = config.exclusions

In [7]:
df_all = df_all.sort([config.encounter_col, config.event_dt_col])

In [8]:
vento2_config

O2DeliveryConfig(mechanical_grouper='O2 Delivery Mechanical Ventilation', mechanical_start_values=['mechanical ventilator', 'ventilator', 'BiPAP', 'CPAP', 'NPPV/NIV'], termination_groupers=['O2 Delivery Nasal Cannula', 'O2 Delivery Simple Face Mask', 'O2 Delivery Room Air'], high_flow_termination_grouper='O2 Delivery High-Flow', high_flow_termination_values=['high-flow nasal cannula', 'high-flow nasal cannula;heated', 'high-flow nasal cannula;humidified', 'humidified;high-flow nasal cannula', 'high-flow mask'], non_rebreather_termination_grouper='O2 Delivery Non-Rebreather Mask', non_rebreather_termination_values=['nonrebreather mask', 'partial rebreather mask', 'blender system'], start_flag_col='mechanical_o2_start_flag', termination_flag_col='o2_delivery_termination_flag')

In [9]:
vent_start_expr = (
	(
		(pl.col(config.grouper_col) == ventonoff_config.grouper)
		&(pl.col(config.raw_val_col).is_in(ventonoff_config.start_values))
	)
	|(
		(pl.col(config.grouper_col) == ventdoc_config.start_grouper)
	)
	|(
		(pl.col(config.grouper_col) == vento2_config.mechanical_grouper)
		&(pl.col(config.raw_val_col).is_in(vento2_config.mechanical_start_values))
	)
)

In [10]:
vento2_config

O2DeliveryConfig(mechanical_grouper='O2 Delivery Mechanical Ventilation', mechanical_start_values=['mechanical ventilator', 'ventilator', 'BiPAP', 'CPAP', 'NPPV/NIV'], termination_groupers=['O2 Delivery Nasal Cannula', 'O2 Delivery Simple Face Mask', 'O2 Delivery Room Air'], high_flow_termination_grouper='O2 Delivery High-Flow', high_flow_termination_values=['high-flow nasal cannula', 'high-flow nasal cannula;heated', 'high-flow nasal cannula;humidified', 'humidified;high-flow nasal cannula', 'high-flow mask'], non_rebreather_termination_grouper='O2 Delivery Non-Rebreather Mask', non_rebreather_termination_values=['nonrebreather mask', 'partial rebreather mask', 'blender system'], start_flag_col='mechanical_o2_start_flag', termination_flag_col='o2_delivery_termination_flag')

In [33]:
ventonoff_config

VentOnOffConfig(grouper='Vent On/Off', start_values=['On Going Hospital Vent', 'Initial', '$ On Going Hospital Vent'], termination_values=['Standby', '$ Extubation'], null_value_terminates=True, start_flag_col='vent_onoff_start_flag', termination_flag_col='vent_onoff_termination_flag')

In [34]:
vent_on_off_termination_values = [
    v for v in ventonoff_config.termination_values if v!='Standby'
]
vent_on_off_termination_values_expr = pl.col(config.raw_val_col).is_in(vent_on_off_termination_values)

if ventonoff_config.null_value_terminates:
    vent_on_off_termination_values_expr = (
		vent_on_off_termination_values_expr | pl.col(config.raw_val_col).is_null()
	)


vent_end_expr = (
	(
		# (pl.col(config.grouper_col) == ventonoff_config.grouper)
		# & ( (pl.col(config.raw_val_col)=='$ Extubation') | pl.col(config.raw_val_col).is_null() )
		(pl.col(config.grouper_col) == ventonoff_config.grouper)
        &vent_on_off_termination_values_expr
	)
	|(
		(pl.col(config.grouper_col) == ventdoc_config.termination_grouper)
	)
	|(
		(pl.col(config.grouper_col) == vento2_config.mechanical_grouper)
		&(pl.col(config.raw_val_col).is_in(vento2_config.termination_groupers))
	)
	|(
		(pl.col(config.grouper_col) == vento2_config.high_flow_termination_grouper)
		&(pl.col(config.raw_val_col).is_in(vento2_config.high_flow_termination_values))
	)
	|(
		(pl.col(config.grouper_col) == vento2_config.non_rebreather_termination_grouper)
		&(pl.col(config.raw_val_col).is_in(vento2_config.non_rebreather_termination_values))
	)
)

In [35]:
df_vent_start_end = df_all.with_columns(
	pl.when(vent_start_expr).then(pl.lit(1)).otherwise(pl.lit(0)).alias("vent_start"),
	pl.when(vent_end_expr).then(pl.lit(1)).otherwise(pl.lit(0)).alias("vent_end"),
).filter( (pl.col("vent_start")>0) | (pl.col("vent_end")>0))

In [36]:
df_vent_start_end.filter(
	((pl.col("vent_start").shift(1).over("EncounterEpicCsn") == 1)&(pl.col("vent_start")==0)& (pl.col("vent_end")==1) )
)

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,sys,dia,map,pf_ratio,PF_Ratio_Flag,vent_start,vent_end
i64,datetime[μs],str,str,str,f64,str,f64,f64,f64,f64,i32,i32,i32
659308243,2022-06-17 13:47:00,"""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ Extubation""",null,null,null,null,null,0,1
659308243,2022-06-25 13:00:00,"""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ Extubation""",null,null,null,null,null,0,1
659308243,2022-06-28 11:07:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""high-flow nasal cannula""",null,null,null,null,null,0,1
659308243,2022-06-30 11:17:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""humidified;high-flow nasal can…",null,null,null,null,null,0,1
659308243,2022-07-02 08:32:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""high-flow nasal cannula;humidi…",null,null,null,null,null,0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…
753113388,2026-05-25 17:45:00,"""Flowsheet""","""Vent off Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""Discontinued""",null,null,null,null,null,0,1
753347320,2026-05-20 19:30:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""high-flow nasal cannula;heated""",null,null,null,null,null,0,1
753347320,2026-05-21 10:35:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""high-flow nasal cannula""",null,null,null,null,null,0,1


In [37]:
df_vent_start_end.group_by(
	"EncounterEpicCsn"
).agg(pl.col("vent_start").first(), pl.col("vent_end").first()).filter(
	(pl.col("vent_end")==1)&
	(pl.col("vent_start")==0)
)

EncounterEpicCsn,vent_start,vent_end
i64,i32,i32
661424247,0,1
662851033,0,1
662905347,0,1
663090810,0,1
663374031,0,1
…,…,…
753170359,0,1
753347320,0,1
753425965,0,1


In [31]:
df_vent_start_end.filter(pl.col("EncounterEpicCsn")==663374031)

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,sys,dia,map,pf_ratio,PF_Ratio_Flag,vent_start,vent_end
i64,datetime[μs],str,str,str,f64,str,f64,f64,f64,f64,i32,i32,i32
663374031,2022-06-30 05:50:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""high-flow nasal cannula;humidi…",null,null,null,null,null,0,1
663374031,2022-06-30 10:09:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""high-flow nasal cannula;humidi…",null,null,null,null,null,0,1
663374031,2022-06-30 10:15:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""high-flow nasal cannula""",null,null,null,null,null,0,1
663374031,2022-06-30 12:45:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""high-flow nasal cannula;humidi…",null,null,null,null,null,0,1
663374031,2022-06-30 17:15:00,"""Flowsheet""","""O2 Delivery Mechanical Ventila…","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""BiPAP""",null,null,null,null,null,1,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…
663374031,2022-09-24 03:05:00,"""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ On Going Hospital Vent""",null,null,null,null,null,1,0
663374031,2022-09-24 03:34:00,"""Flowsheet""","""O2 Delivery Mechanical Ventila…","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""ventilator""",null,null,null,null,null,1,0
663374031,2022-09-24 07:15:00,"""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ On Going Hospital Vent""",null,null,null,null,null,1,0


In [28]:
df_vent_start_end.filter(pl.col("EncounterEpicCsn")==661424247)

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,sys,dia,map,pf_ratio,PF_Ratio_Flag,vent_start,vent_end
i64,datetime[μs],str,str,str,f64,str,f64,f64,f64,f64,i32,i32,i32
661424247,2022-08-28 02:18:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""high-flow nasal cannula""",null,null,null,null,null,0,1
661424247,2022-08-28 04:41:00,"""Flowsheet""","""O2 Delivery Non-Rebreather Mas…","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""nonrebreather mask""",null,null,null,null,null,0,1
661424247,2022-08-28 05:15:00,"""Flowsheet""","""Vent on Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ Initial""",null,null,null,null,null,1,0
661424247,2022-08-28 05:55:00,"""Flowsheet""","""O2 Delivery Mechanical Ventila…","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""ventilator""",null,null,null,null,null,1,0
661424247,2022-08-28 07:00:00,"""Flowsheet""","""O2 Delivery Mechanical Ventila…","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""ventilator""",null,null,null,null,null,1,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…
661424247,2022-09-09 10:22:00,"""Flowsheet""","""O2 Delivery Mechanical Ventila…","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""ventilator""",null,null,null,null,null,1,0
661424247,2022-09-09 20:17:00,"""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ On Going Hospital Vent""",null,null,null,null,null,1,0
661424247,2022-09-09 23:06:00,"""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ On Going Hospital Vent""",null,null,null,null,null,1,0


In [43]:
df_vent_start_end.with_columns(
	(pl.col("vent_start").shift(1).over("EncounterEpicCsn") != pl.col("vent_start")).alias("vent_shift")
).with_columns(pl.col("vent_shift").fill_null(False)).with_columns(
	pl.col("vent_shift").cum_sum().over("EncounterEpicCsn").alias("episodes")
).group_by("EncounterEpicCsn", "episodes").agg(pl.col("Event_DateTime").min().alias("begin_dt"), pl.col("Event_DateTime").max().alias("end_dt"), pl.col("Value"), pl.col("vent_start").first().alias('Vent_on'), pl.col("vent_start").len().alias('n_rows')).sort(by=['EncounterEpicCsn', 'episodes'])

EncounterEpicCsn,episodes,begin_dt,end_dt,Value,Vent_on,n_rows
i64,u32,datetime[μs],datetime[μs],list[str],i32,u64
659308243,0,2022-06-14 21:51:00,2022-06-17 12:40:00,"[""$ Initial"", ""$ On Going Hospital Vent"", … ""$ On Going Hospital Vent""]",1,22
659308243,1,2022-06-17 13:47:00,2022-06-17 13:47:00,"[""$ Extubation""]",0,1
659308243,2,2022-06-17 19:54:00,2022-06-25 11:00:00,"[""$ Initial"", ""$ On Going Hospital Vent"", … ""$ On Going Hospital Vent""]",1,53
659308243,3,2022-06-25 13:00:00,2022-06-25 13:00:00,"[""$ Extubation""]",0,1
659308243,4,2022-06-28 07:08:00,2022-06-28 07:08:00,"[""BiPAP""]",1,1
…,…,…,…,…,…,…
753371568,1,2026-05-26 11:00:00,2026-05-26 18:05:00,"[null, null, ""Discontinued""]",0,3
753425965,0,2026-05-24 17:08:00,2026-05-28 03:00:00,"[""nonrebreather mask"", ""high-flow nasal cannula;humidified"", … ""high-flow nasal cannula;humidified""]",0,18
753498904,0,2026-05-22 12:46:00,2026-05-22 12:50:00,"[""nonrebreather mask"", ""nonrebreather mask""]",0,2


In [18]:
df_vent_null_ventonoff = df_vent_start_end.filter(
	(pl.col("Event_Grouper") == "Vent On/Off")&
	pl.col("Value").is_null()
)

In [24]:
df_vent_null_preceeded_by_vent = df_vent_null_ventonoff.select("EncounterEpicCsn", "Event_DateTime", "Event_Grouper", "Value").join_asof(
    df_vent_start_end.filter(
        	~((pl.col("Event_Grouper") == "Vent On/Off")&
			pl.col("Value").is_null())
	).select("EncounterEpicCsn", pl.col("Event_DateTime").alias('Event_DateTime_all'), "Event_Grouper", "Value"), 
	left_on = "Event_DateTime",
	right_on = "Event_DateTime_all",
	by="EncounterEpicCsn",
    strategy="backward",
    suffix='all'
)

/tmp/ipykernel_2819740/1576885736.py:1: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df_vent_null_preceeded_by_vent = df_vent_null_ventonoff.select("EncounterEpicCsn", "Event_DateTime", "Event_Grouper", "Value").join_asof(


In [61]:
df_vent_null_preceeded_by_vent.with_columns(
    (pl.col("Event_DateTime")-pl.col("Event_DateTime_all")).dt.total_hours(fractional=True).alias("hours_from_vent_to_null")
).filter(
    pl.col("hours_from_vent_to_null") == 0
)

EncounterEpicCsn,Event_DateTime,Event_Grouper,Value,Event_DateTime_all,Event_Grouperall,Valueall,hours_from_vent_to_null
i64,datetime[μs],str,str,datetime[μs],str,str,f64
664211059,2022-09-19 00:55:00,"""Vent On/Off""",null,2022-09-19 00:55:00,"""O2 Delivery Mechanical Ventila…","""ventilator""",0.0
667146450,2022-08-25 07:00:00,"""Vent On/Off""",null,2022-08-25 07:00:00,"""O2 Delivery Mechanical Ventila…","""ventilator""",0.0
669821634,2022-10-21 00:21:00,"""Vent On/Off""",null,2022-10-21 00:21:00,"""O2 Delivery Mechanical Ventila…","""ventilator""",0.0
673279459,2022-12-04 13:55:00,"""Vent On/Off""",null,2022-12-04 13:55:00,"""O2 Delivery Mechanical Ventila…","""ventilator""",0.0
673348701,2022-12-15 15:30:00,"""Vent On/Off""",null,2022-12-15 15:30:00,"""O2 Delivery Mechanical Ventila…","""ventilator""",0.0
…,…,…,…,…,…,…,…
739776319,2025-11-10 07:40:00,"""Vent On/Off""",null,2025-11-10 07:40:00,"""O2 Delivery Mechanical Ventila…","""ventilator""",0.0
744908255,2026-03-20 19:00:00,"""Vent On/Off""",null,2026-03-20 19:00:00,"""O2 Delivery Mechanical Ventila…","""ventilator""",0.0
746128968,2026-02-26 12:40:00,"""Vent On/Off""",null,2026-02-26 12:40:00,"""O2 Delivery Mechanical Ventila…","""ventilator""",0.0


In [ ]:
d = {}
for rows in df_vent_null_preceeded_by_vent.iter_rows(named=True):
    d[(rows['Event_Grouperall'], rows['Valueall'])] = d.get((rows['Event_Grouperall'], rows['Valueall']), 0)+1
    


In [32]:
sorted(d.items(), key=lambda kv: -kv[1])

[(('Vent On/Off', '$ On Going Hospital Vent'), 497),
 (('O2 Delivery Mechanical Ventilation', 'ventilator'), 294),
 (('Vent On/Off', 'On Going Hospital Vent'), 130),
 (('Vent off Documentation', 'Discontinued'), 31),
 ((None, None), 23),
 (('Vent on Documentation', '$ Initial'), 15),
 (('O2 Delivery Mechanical Ventilation', 'BiPAP'), 14),
 (('O2 Delivery High-Flow', 'humidified;high-flow nasal cannula'), 9),
 (('O2 Delivery High-Flow', 'high-flow nasal cannula'), 7),
 (('O2 Delivery Non-Rebreather Mask', 'nonrebreather mask'), 6),
 (('O2 Delivery High-Flow', 'high-flow nasal cannula;humidified'), 4),
 (('Vent On/Off', '$ Extubation'), 3),
 (('O2 Delivery High-Flow', 'high-flow nasal cannula;heated'), 3),
 (('Vent On/Off', 'Initial'), 3),
 (('O2 Delivery Mechanical Ventilation', 'NPPV/NIV'), 2)]

### Continue with the AI analysis

In [40]:
df_vent_start = (
    df_vent_start_end.filter(
		pl.col("vent_start") == 1
	).select(
        config.encounter_col,
        pl.col("Event_DateTime").alias("vent_start_dt"),
        pl.col("Event_Grouper").alias("vent_start_grouper"),
        pl.col("Value").alias("vent_start_value"),
	)
    .sort(by=[config.encounter_col, 'vent_start_dt'])
	.with_row_index("vent_start_evidence_id", offset=1)
)
df_vent_start

vent_start_evidence_id,EncounterEpicCsn,vent_start_dt,vent_start_grouper,vent_start_value
u64,i64,datetime[μs],str,str
1,659308243,2022-06-14 21:51:00,"""Vent on Documentation""","""$ Initial"""
2,659308243,2022-06-14 22:36:00,"""Vent On/Off""","""$ On Going Hospital Vent"""
3,659308243,2022-06-15 00:12:00,"""Vent On/Off""","""$ On Going Hospital Vent"""
4,659308243,2022-06-15 02:55:00,"""Vent On/Off""","""$ On Going Hospital Vent"""
5,659308243,2022-06-15 04:54:00,"""Vent On/Off""","""$ On Going Hospital Vent"""
…,…,…,…,…
268590,753558469,2026-05-24 19:21:00,"""O2 Delivery Mechanical Ventila…","""BiPAP"""
268591,753558469,2026-05-24 21:15:00,"""O2 Delivery Mechanical Ventila…","""BiPAP"""
268592,753558469,2026-05-27 19:18:00,"""O2 Delivery Mechanical Ventila…","""BiPAP"""


In [ ]:
df_vent_end = (
    df_vent_start_end
    .filter(pl.col("vent_end") == 1)
	 .group_by(config.encounter_col, config.event_dt_col)
    .agg(
        pl.col(config.grouper_col)
        .unique()
        .sort()
        .alias("vent_termination_groupers"),

        pl.col(config.raw_val_col)
        .drop_nulls()
        .unique()
        .sort()
        .alias("vent_termination_values"),
    )
    .rename({config.event_dt_col: "vent_termination_dt"})
    .sort(config.encounter_col, "vent_termination_dt")
)
df_vent_end.select(
	pl.col('vent_termination_groupers').list.len()
)['vent_termination_groupers'].max()

2

In [48]:
df_vent_end.filter(
	pl.col('vent_termination_groupers').list.len() == 2
)

EncounterEpicCsn,vent_termination_dt,vent_termination_groupers,vent_termination_values
i64,datetime[μs],list[str],list[str]
663374031,2022-07-25 10:39:00,"[""O2 Delivery High-Flow"", ""Vent On/Off""]","[""$ Extubation"", ""high-flow nasal cannula;humidified""]"
663879519,2022-07-03 11:00:00,"[""O2 Delivery High-Flow"", ""Vent off Documentation""]","[""Discontinued"", ""high-flow nasal cannula""]"
664010526,2022-07-03 15:50:00,"[""O2 Delivery High-Flow"", ""Vent off Documentation""]","[""Discontinued"", ""high-flow nasal cannula""]"
664754198,2022-07-22 12:30:00,"[""O2 Delivery High-Flow"", ""Vent off Documentation""]","[""Discontinued"", ""high-flow nasal cannula;heated""]"
667686916,2022-09-06 09:45:00,"[""O2 Delivery High-Flow"", ""Vent off Documentation""]","[""Discontinued"", ""high-flow nasal cannula""]"
…,…,…,…
747154189,2026-02-25 07:45:00,"[""O2 Delivery High-Flow"", ""Vent off Documentation""]","[""Discontinued"", ""high-flow nasal cannula""]"
747410809,2026-03-14 17:23:00,"[""O2 Delivery High-Flow"", ""Vent off Documentation""]","[""Discontinued"", ""high-flow nasal cannula""]"
748831725,2026-03-24 12:55:00,"[""O2 Delivery High-Flow"", ""Vent off Documentation""]","[""Discontinued"", ""high-flow nasal cannula;humidified""]"


In [50]:
df_vent_start_end.filter(
    (pl.col("EncounterEpicCsn") == 748831725)&
    (pl.col("Event_DateTime") == dt.datetime(2026, 3, 24, 12, 55))
)

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,sys,dia,map,pf_ratio,PF_Ratio_Flag,vent_start,vent_end
i64,datetime[μs],str,str,str,f64,str,f64,f64,f64,f64,i32,i32,i32
748831725,2026-03-24 12:55:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""high-flow nasal cannula;humidi…",null,null,null,null,null,0,1
748831725,2026-03-24 12:55:00,"""Flowsheet""","""Vent off Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""Discontinued""",null,null,null,null,null,0,1


In [92]:
df_vent_start_end.group_by(
    "EncounterEpicCsn", "Event_DateTime"
).agg(
	pl.col("vent_start").unique().sort(),
	pl.col("vent_end").unique().sort(),
).filter(
    (pl.col("vent_start").list.len()>1)
    |
    (pl.col("vent_end").list.len()>1)
).filter(
    pl.col("EncounterEpicCsn") == 663374031
)

EncounterEpicCsn,Event_DateTime,vent_start,vent_end
i64,datetime[μs],list[i32],list[i32]


In [56]:
df_vent_start_end.filter(
    (pl.col("EncounterEpicCsn") == 736836690)
    &(pl.col("Event_DateTime") == dt.datetime(2025, 10, 31, 21))
)

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,sys,dia,map,pf_ratio,PF_Ratio_Flag,vent_start,vent_end
i64,datetime[μs],str,str,str,f64,str,f64,f64,f64,f64,i32,i32,i32
736836690,2025-10-31 21:00:00,"""Flowsheet""","""O2 Delivery Mechanical Ventila…","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""ventilator""",null,null,null,null,null,1,0
736836690,2025-10-31 21:00:00,"""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,null,null,null,null,null,null,0,1


In [78]:
df_vent_start_end.group_by(
    "EncounterEpicCsn", "Event_DateTime"
).agg(
	pl.col("vent_start").unique().sort(),
	pl.col("vent_end").unique().sort(),
).filter(
    (pl.col("vent_start").list.len()>1)
    |
    (pl.col("vent_end").list.len()>1)
).select(
    pl.col("vent_start").list.len().max().alias('max_vent_start_len'),
    pl.col("vent_end").list.len().max().alias('max_vent_end_len')
)

max_vent_start_len,max_vent_end_len
u64,u64
2,2


In [79]:
df_on_off_vent_0 = df_vent_start_end.group_by(
    "EncounterEpicCsn", "Event_DateTime"
).agg(
	pl.col("vent_start").unique().sort(),
	pl.col("vent_end").unique().sort(),
).filter(
    (pl.col("vent_start").list.len()>1)
    |
    (pl.col("vent_end").list.len()>1)
)

df_0_vent2null = df_vent_null_preceeded_by_vent.with_columns(
    (pl.col("Event_DateTime")-pl.col("Event_DateTime_all")).dt.total_hours(fractional=True).alias("hours_from_vent_to_null")
).filter(
    pl.col("hours_from_vent_to_null") == 0
).select("EncounterEpicCsn", "Event_DateTime")

In [80]:
df_on_off_vent_0_not_null = df_on_off_vent_0.join(df_0_vent2null, how='anti', on=['EncounterEpicCsn', 'Event_DateTime'])

In [88]:
d_grouper = {}
for row in df_on_off_vent_0_not_null.iter_rows(named=True):
	df_sub = (df_vent_start_end.filter(
		(pl.col("EncounterEpicCsn") == row[config.encounter_col])
		&(pl.col("Event_DateTime") == row[config.event_dt_col])
	))
	key1 = (df_sub[0]['Event_Grouper'].item(), df_sub[0]['Value'].item(),
		df_sub[1]['Event_Grouper'].item(), df_sub[1]['Value'].item() )
	key2 = (df_sub[1]['Event_Grouper'].item(), df_sub[1]['Value'].item(),
		df_sub[0]['Event_Grouper'].item(), df_sub[0]['Value'].item() )
	d_grouper[key1] = d_grouper.get(key1, 0) + 1
	d_grouper[key2] = d_grouper.get(key2, 0) + 1

In [93]:
df_on_off_vent_0_not_null	

EncounterEpicCsn,Event_DateTime,vent_start,vent_end
i64,datetime[μs],list[i32],list[i32]
711743163,2024-10-28 13:00:00,"[0, 1]","[0, 1]"
744112293,2026-02-16 13:00:00,"[0, 1]","[0, 1]"
698435519,2024-02-21 10:33:00,"[0, 1]","[0, 1]"
699388833,2024-03-08 09:48:00,"[0, 1]","[0, 1]"
734482652,2025-09-01 17:50:00,"[0, 1]","[0, 1]"
…,…,…,…
747431997,2026-03-04 14:05:00,"[0, 1]","[0, 1]"
705075651,2024-06-09 09:35:00,"[0, 1]","[0, 1]"
721527751,2025-03-12 11:50:00,"[0, 1]","[0, 1]"


In [91]:
sorted(d_grouper.items(), key=lambda kv: -kv[1])

[(('Vent off Documentation',
   'Discontinued',
   'O2 Delivery Mechanical Ventilation',
   'BiPAP'),
  28),
 (('O2 Delivery Mechanical Ventilation',
   'BiPAP',
   'Vent off Documentation',
   'Discontinued'),
  28),
 (('Vent off Documentation',
   'Discontinued',
   'O2 Delivery Mechanical Ventilation',
   'ventilator'),
  7),
 (('O2 Delivery Mechanical Ventilation',
   'ventilator',
   'Vent off Documentation',
   'Discontinued'),
  7),
 (('O2 Delivery High-Flow',
   'high-flow nasal cannula',
   'Vent On/Off',
   '$ On Going Hospital Vent'),
  5),
 (('Vent On/Off',
   '$ On Going Hospital Vent',
   'O2 Delivery High-Flow',
   'high-flow nasal cannula'),
  5),
 (('O2 Delivery Mechanical Ventilation',
   'NPPV/NIV',
   'Vent off Documentation',
   'Discontinued'),
  4),
 (('Vent off Documentation',
   'Discontinued',
   'O2 Delivery Mechanical Ventilation',
   'NPPV/NIV'),
  4),
 (('Vent on Documentation',
   '$ Initial',
   'O2 Delivery Non-Rebreather Mask',
   'nonrebreather mask')

In [86]:
df_sub

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,sys,dia,map,pf_ratio,PF_Ratio_Flag,vent_start,vent_end
i64,datetime[μs],str,str,str,f64,str,f64,f64,f64,f64,i32,i32,i32
711743163,2024-10-28 13:00:00,"""Flowsheet""","""Vent off Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""Discontinued""",null,null,null,null,null,0,1
711743163,2024-10-28 13:00:00,"""Flowsheet""","""O2 Delivery Mechanical Ventila…","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""BiPAP""",null,null,null,null,null,1,0


In [ ]:
df_anti_vent_start_end = df_vent_start_end.join(
    df_on_off_vent_0.select("EncounterEpicCsn", "Event_DateTime"),
	on=['EncounterEpicCsn', 'Event_DateTime'],
    how='anti'
)

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,sys,dia,map,pf_ratio,PF_Ratio_Flag,vent_start,vent_end
i64,datetime[μs],str,str,str,f64,str,f64,f64,f64,f64,i32,i32,i32
659308243,2022-06-14 21:51:00,"""Flowsheet""","""Vent on Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ Initial""",null,null,null,null,null,1,0
659308243,2022-06-14 22:36:00,"""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ On Going Hospital Vent""",null,null,null,null,null,1,0
659308243,2022-06-15 00:12:00,"""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ On Going Hospital Vent""",null,null,null,null,null,1,0
659308243,2022-06-15 02:55:00,"""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ On Going Hospital Vent""",null,null,null,null,null,1,0
659308243,2022-06-15 04:54:00,"""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ On Going Hospital Vent""",null,null,null,null,null,1,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…
753558469,2026-05-24 19:21:00,"""Flowsheet""","""O2 Delivery Mechanical Ventila…","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""BiPAP""",null,null,null,null,null,1,0
753558469,2026-05-24 21:15:00,"""Flowsheet""","""O2 Delivery Mechanical Ventila…","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""BiPAP""",null,null,null,null,null,1,0
753558469,2026-05-27 19:18:00,"""Flowsheet""","""O2 Delivery Mechanical Ventila…","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""BiPAP""",null,null,null,null,null,1,0


In [103]:
df_on_off_vent_0_not_null.join_asof(
    df_anti_vent_start_end.select(
        config.encounter_col,
        pl.col(config.event_dt_col).alias('start_or_end_dt'),
        config.grouper_col,
        config.raw_val_col
	),
    left_on="Event_DateTime",
	right_on="start_or_end_dt",
	by='EncounterEpicCsn',
    strategy='forward'
).filter(
    pl.col("Event_Grouper").is_not_null()
)

/tmp/ipykernel_2819740/1045624455.py:1: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df_on_off_vent_0_not_null.join_asof(


EncounterEpicCsn,Event_DateTime,vent_start,vent_end,start_or_end_dt,Event_Grouper,Value
i64,datetime[μs],list[i32],list[i32],datetime[μs],str,str
711743163,2024-10-28 13:00:00,"[0, 1]","[0, 1]",2024-10-31 08:47:00,"""O2 Delivery High-Flow""","""high-flow nasal cannula"""
744112293,2026-02-16 13:00:00,"[0, 1]","[0, 1]",2026-02-16 14:48:00,"""O2 Delivery Mechanical Ventila…","""BiPAP"""
698435519,2024-02-21 10:33:00,"[0, 1]","[0, 1]",2024-02-21 11:10:00,"""O2 Delivery Mechanical Ventila…","""ventilator"""
699388833,2024-03-08 09:48:00,"[0, 1]","[0, 1]",2024-03-08 10:56:00,"""O2 Delivery Mechanical Ventila…","""BiPAP"""
693573011,2023-11-27 07:00:00,"[0, 1]","[0, 1]",2023-11-27 10:15:00,"""O2 Delivery Mechanical Ventila…","""BiPAP"""
…,…,…,…,…,…,…
674367576,2022-12-28 13:25:00,"[0, 1]","[0, 1]",2022-12-29 08:06:00,"""O2 Delivery Mechanical Ventila…","""BiPAP"""
747431997,2026-03-04 14:05:00,"[0, 1]","[0, 1]",2026-03-04 15:06:00,"""O2 Delivery Mechanical Ventila…","""BiPAP"""
705075651,2024-06-09 09:35:00,"[0, 1]","[0, 1]",2024-06-09 13:15:00,"""O2 Delivery Mechanical Ventila…","""ventilator"""


In [106]:
df_vent_start_end.filter(
    (pl.col("EncounterEpicCsn") == 674367576)
    &(pl.col("Event_DateTime") == dt.datetime(2022, 12, 28, 13, 25))
)

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,sys,dia,map,pf_ratio,PF_Ratio_Flag,vent_start,vent_end
i64,datetime[μs],str,str,str,f64,str,f64,f64,f64,f64,i32,i32,i32
674367576,2022-12-28 13:25:00,"""Flowsheet""","""O2 Delivery Mechanical Ventila…","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""BiPAP""",null,null,null,null,null,1,0
674367576,2022-12-28 13:25:00,"""Flowsheet""","""Vent off Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""Discontinued""",null,null,null,null,null,0,1


In [107]:
df_vent_start_end.filter(
    (pl.col("EncounterEpicCsn") == 705075651)
    # &(pl.col("Event_DateTime") == dt.datetime(2024, 12, 28, 13, 25))
)

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,sys,dia,map,pf_ratio,PF_Ratio_Flag,vent_start,vent_end
i64,datetime[μs],str,str,str,f64,str,f64,f64,f64,f64,i32,i32,i32
705075651,2024-06-06 18:20:00,"""Flowsheet""","""Vent on Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ Initial""",null,null,null,null,null,1,0
705075651,2024-06-06 19:15:00,"""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ On Going Hospital Vent""",null,null,null,null,null,1,0
705075651,2024-06-06 23:10:00,"""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ On Going Hospital Vent""",null,null,null,null,null,1,0
705075651,2024-06-07 03:15:00,"""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ On Going Hospital Vent""",null,null,null,null,null,1,0
705075651,2024-06-07 08:10:00,"""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ On Going Hospital Vent""",null,null,null,null,null,1,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…
705075651,2024-06-15 10:42:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""high-flow nasal cannula;humidi…",null,null,null,null,null,0,1
705075651,2024-06-15 14:29:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""high-flow nasal cannula;humidi…",null,null,null,null,null,0,1
705075651,2024-06-15 15:59:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""high-flow nasal cannula""",null,null,null,null,null,0,1
